# Jute Leaf Disease Classification with Transfer Learning (ResNet-18)

This notebook fine-tunes a pre-trained **ResNet-18** model on the **Project-AgML/jute_disease_classification** dataset (5 categories: `Dieback`, `Fresh`, `Holed`, `Mosaic`, `Stem Soft Rot`) using **PyTorch**, **torchvision**, and **Matplotlib**.

### Pipeline Steps
1. Load the local Arrow dataset (`data/jute_disease_classification-train.arrow`) with a stratified 80/20 train/validation split.
2. Apply data augmentation and ImageNet normalization from `src/jute_disease_classifier/transform.py`.
3. Fine-tune ResNet-18 (`src/jute_disease_classifier/model.py`) with `CrossEntropyLoss`, `AdamW`, and `ReduceLROnPlateau`.
4. Track train/validation loss and accuracy, save checkpoints, and plot loss/accuracy curves & confusion matrix (`src/jute_disease_classifier/utils.py`).
5. Run sample inference on validation images.

In [ ]:
import time
from collections import Counter

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset as HFDataset, load_dataset
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from src.jute_disease_classifier import (
    build_resnet18,
    config,
    count_parameters,
    get_train_transforms,
    get_val_transforms,
    load_checkpoint,
    plot_confusion_matrix,
    plot_loss_curves,
    save_checkpoint,
    save_history,
    set_seed,
)

set_seed(config.SEED)
print(f"PyTorch version: {torch.__version__}")
print(f"Using device:    {config.DEVICE}")
print(f"Classes ({config.NUM_CLASSES}):     {config.CLASS_NAMES}")

In [ ]:
# Load local Arrow dataset if available, otherwise fall back to Hugging Face Hub
if config.LOCAL_ARROW_FILE.exists():
    print(f"Loading local dataset from: {config.LOCAL_ARROW_FILE}")
    raw_dataset = HFDataset.from_file(str(config.LOCAL_ARROW_FILE))
else:
    print(f"Local Arrow file not found; downloading {config.HF_DATASET_NAME}...")
    raw_dataset = load_dataset(config.HF_DATASET_NAME, split="train")

class_names = raw_dataset.features["label"].names
print(f"Total images: {len(raw_dataset)}")

# Class distribution
counts = Counter(raw_dataset["label"])
for idx, name in enumerate(class_names):
    print(f"  [{idx}] {name:15s}: {counts[idx]} images")

# Stratified 80/20 train/validation split
splits = raw_dataset.train_test_split(
    test_size=config.VAL_SPLIT_RATIO,
    seed=config.SEED,
    stratify_by_column="label",
)
print(f"\nTrain split: {len(splits['train'])} | Val split: {len(splits['test'])}")

In [ ]:
# Display one sample image per class
fig, axes = plt.subplots(1, len(class_names), figsize=(16, 3.5))
seen_labels = {}
for sample in splits["train"]:
    lbl = sample["label"]
    if lbl not in seen_labels:
        seen_labels[lbl] = sample["image"]
    if len(seen_labels) == len(class_names):
        break

for idx, ax in enumerate(axes):
    ax.imshow(seen_labels[idx])
    ax.set_title(class_names[idx], fontsize=12, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class JuteLeafDataset(Dataset):
    """PyTorch Dataset wrapper around a Hugging Face dataset split."""

    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item["image"]
        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        if image.mode != "RGB":
            image = image.convert("RGB")
        label = int(item["label"])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


train_dataset = JuteLeafDataset(splits["train"], transform=get_train_transforms(config.IMAGE_SIZE))
val_dataset = JuteLeafDataset(splits["test"], transform=get_val_transforms(config.IMAGE_SIZE))

pin_memory = torch.cuda.is_available()
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=pin_memory,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
model = build_resnet18(
    num_classes=len(class_names),
    pretrained=True,
    freeze_backbone=config.FREEZE_BACKBONE,
    dropout_rate=config.DROPOUT_RATE,
).to(config.DEVICE)

trainable_params, total_params = count_parameters(model)
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({trainable_params / total_params * 100:.1f}%)")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device, epoch):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch:2d} [Train]", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix({"Loss": f"{running_loss / total:.4f}", "Acc": f"{correct / total * 100:.2f}%"})
    return running_loss / total, (correct / total) * 100.0


def validate(model, dataloader, criterion, device, epoch=0):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds = [], []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch:2d} [Val]  ", leave=False)
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            pbar.set_postfix({"Loss": f"{running_loss / total:.4f}", "Acc": f"{correct / total * 100:.2f}%"})
    return running_loss / total, (correct / total) * 100.0, all_labels, all_preds


history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_acc = 0.0
best_val_loss = float("inf")
best_labels, best_preds = [], []
best_ckpt = config.CHECKPOINT_DIR / "best_resnet18.pth"
last_ckpt = config.CHECKPOINT_DIR / "last_resnet18.pth"

# --- Stop Triggers ---
EARLY_STOPPING_PATIENCE = 3   # Stop if no val improvement for N consecutive epochs
TARGET_VAL_ACC = 98.5         # Stop immediately once val_acc >= target (set None to disable)
no_improve_epochs = 0

start_time = time.time()
for epoch in range(1, config.NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, config.DEVICE, epoch)
    val_loss, val_acc, val_labels, val_preds = validate(model, val_loader, criterion, config.DEVICE, epoch)

    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(round(train_loss, 4))
    history["train_acc"].append(round(train_acc, 2))
    history["val_loss"].append(round(val_loss, 4))
    history["val_acc"].append(round(val_acc, 2))

    improved = (val_acc > best_val_acc) or (val_acc == best_val_acc and val_loss < best_val_loss)
    if improved or epoch == 1:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_labels, best_preds = val_labels, val_preds
        save_checkpoint(model, optimizer, epoch, val_acc, val_loss, class_names, best_ckpt)
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1

    save_checkpoint(model, optimizer, epoch, val_acc, val_loss, class_names, last_ckpt)
    save_history(history, config.OUTPUTS_DIR / "training_history.json")

    badge = " [*Best]" if improved else ""
    print(
        f"Epoch [{epoch:2d}/{config.NUM_EPOCHS:2d}] | "
        f"Train Loss: {train_loss:.4f} - Train Acc: {train_acc:6.2f}% | "
        f"Val Loss: {val_loss:.4f} - Val Acc: {val_acc:6.2f}% | "
        f"LR: {current_lr:.1e} | Time: {time.time() - t0:5.1f}s{badge}"
    )

    # Check stop triggers
    if TARGET_VAL_ACC is not None and val_acc >= TARGET_VAL_ACC:
        print(f"\n[STOP TRIGGER] Reached target validation accuracy ({val_acc:.2f}% >= {TARGET_VAL_ACC}%). Stopping training.")
        break
    if no_improve_epochs >= EARLY_STOPPING_PATIENCE:
        print(f"\n[EARLY STOPPING] No validation improvement for {EARLY_STOPPING_PATIENCE} consecutive epochs. Stopping training.")
        break

print(f"\nTraining finished in {(time.time() - start_time) / 60:.2f} min. Best Val Acc: {best_val_acc:.2f}% (Val Loss: {best_val_loss:.4f})")

In [ ]:
# Plot and save Loss & Accuracy curves
plot_loss_curves(
    history,
    save_path=config.PLOTS_DIR / "loss_accuracy_curves.png",
    show_plot=True,
)

# Plot and save Normalized Confusion Matrix
plot_confusion_matrix(
    best_labels,
    best_preds,
    class_names,
    save_path=config.PLOTS_DIR / "confusion_matrix.png",
    show_plot=True,
)

In [ ]:
# Load best checkpoint and run inference on 5 validation samples
eval_model = build_resnet18(num_classes=len(class_names), pretrained=False).to(config.DEVICE)
load_checkpoint(best_ckpt, eval_model, device=config.DEVICE)
eval_model.eval()

val_tf = get_val_transforms(config.IMAGE_SIZE)
fig, axes = plt.subplots(1, 5, figsize=(16, 3.8))

for idx, ax in enumerate(axes):
    sample = splits["test"][idx * (len(splits["test"]) // 5)]
    img = sample["image"].convert("RGB")
    true_label = class_names[sample["label"]]

    tensor = val_tf(img).unsqueeze(0).to(config.DEVICE)
    with torch.no_grad():
        probs = F.softmax(eval_model(tensor), dim=1)[0]
    pred_idx = int(probs.argmax())
    pred_label = class_names[pred_idx]
    conf = probs[pred_idx].item() * 100.0

    color = "green" if pred_label == true_label else "red"
    ax.imshow(img)
    ax.set_title(f"Pred: {pred_label} ({conf:.1f}%)\nTrue: {true_label}", color=color, fontsize=10, fontweight="bold")
    ax.axis("off")

plt.tight_layout()
plt.show()